# PRO3342 Modelagem e Simulação de Sistemas de Produção
## Projeto BVE - Modelo computacional: Navios

### Grupo G112

1. Francisco Rocha Freitas Brandão (18404341)
2. Inigo Anduaga (18404466)
3. Kepa Danis (18404719)
4. Vasco Oliveira (18440540)

Neste lab, o modelo conceitual desenvolvido no Lab 07 constitui a referência para a implementação do modelo computacional dos navios.

## 1. Objetivo

O modelo implementa o **submodelo 1 (Navios)** do modelo conceitual do grupo (`G112_Modelo_Conceitual`, versão 1.1, seções 2.1, 4 e 5). Ele representa a operação dos navios no Porto Mar Azul e gera a demanda de combustível que alimentará os submodelos das barcaças e do PCO. Cada terminal *j* atende uma classe de navios e tem *c_j* berços equivalentes; o porto é simulado como *M* filas M/G/*c_j* independentes: notificação com chegadas de Poisson, chegada ao fundeio após a antecedência, espera FIFO, manobra de entrada, operação, manobra de saída e saída do porto.

Pela premissa central do modelo conceitual, o abastecimento não altera a operação do navio, que deixa o porto ao fim da operação, abastecido ou não. Por isso este submodelo roda de forma independente das barcaças e do PCO.

| | Arquivo | Uso neste modelo |
|:--|:--|:--|
| Entrada | `entrada/terminais.csv` | Parâmetros de cada terminal / classe de navio |
| Entrada | `entrada/cenario.csv` | Data de início, horizonte, replicações, semente e fator de demanda *f_d* |
| Entrada | `entrada/distancias.csv` | Preparado para o submodelo 2; não é lido aqui |
| Saída | `saida/log_navios.csv` | Uma linha por navio; entrada do PCO no submodelo 3 |

`terminais.csv` e `distancias.csv` são gerados a partir dos dados do cenário-base (`entrada/D08_2_BVE_MComp_Navios_-_Dados.md`). `cenario.csv` segue o formato do modelo conceitual, com os campos do modelo de referência e os acréscimos do grupo (`fator_demanda`, `horarios_programacao`, `horizonte_programacao`, `regra_pco`); os três últimos só serão usados pelo PCO.

**Decisões não definidas nos dados**, registradas em `cenario.csv` ou no código:

1. `data_inicio` = 01/01/2026 00:00, apenas para converter o relógio (h) em data-hora.
2. Horizonte de 365 dias, conforme esta atividade (o modelo conceitual prevê 500 dias para os experimentos), uma replicação e semente 20260.
3. `fator_demanda` = 1,0 no cenário-base. Conforme o modelo conceitual, *f_d* multiplica `prob_compra` (limitado a 1) e não altera o tráfego de navios.
4. O porto inicia vazio, sem aquecimento; as primeiras chegadas ocorrem após 48 h.
5. Entram no log os navios que concluem a manobra de saída até o fim do horizonte, como no pseudocódigo do modelo conceitual; os navios ainda em curso são contados à parte na seção 4.
6. `id_navio` sequencial por ordem de notificação (`N000001`, ...), único por cenário e replicação.
7. Além das distribuições da convenção (uniforme, exponencial, triangular, erlang, lognormal), o código aceita `constante`, usada só nos testes determinísticos.

## 2. Prompt

**Ferramenta de IA:** Claude (Anthropic), em interface de chat. A IA apoiou a preparação das entradas, a escrita do modelo SimPy e dos testes de verificação e a montagem deste notebook. O prompt original foi redigido em espanhol; abaixo está a versão em português, consolidada e atualizada para o modelo conceitual v1.1.

```markdown
Atue como especialista em simulação de eventos discretos (SimPy). Implemente a atividade D08 - Projeto BVE: Modelo Computacional dos Navios (PRO3342).

OBJETIVO
Implementar o submodelo 1 (Navios) do modelo conceitual do grupo G112 (versão 1.1): operação dos navios
no Porto Mar Azul e geração da demanda de combustível. Não implementar barcaças, PCO nem abastecimento.

FONTES (ordem de prioridade)
1. Modelo conceitual do grupo G112, versão 1.1 (seções 2.1, 4, 5 e 6), que segue o modelo de referência do professor.
2. Instruções de D08. 3. Dados do cenário-base (arquivo D08_2 em Markdown).
4. Suposições próprias só se imprescindíveis, sempre documentadas. Se algo não estiver definido, indicar antes de assumir.

ENTRADAS (arquivos CSV; nenhum parâmetro do cenário fixo no código)
- terminais.csv: id_terminal, id_classe, n_bercos, taxa_chegada (navios/h), antecedencia (h), tempo_manobra (h),
  dist_operacao e op_p1..p3 (h), prob_compra, dist_volume e vl_p1..p3 (t). Gerar a partir do arquivo de dados.
- cenario.csv: id_cenario, data_inicio, horizonte (dias), n_rep, semente, fator_demanda, horarios_programacao,
  horizonte_programacao, regra_pco (os três últimos só para o PCO).
- distancias.csv (formato longo: origem, destino, distancia; inclui a base): gerar, mas não usar neste modelo.
- Convenção: uniforme(min, max), exponencial(média), triangular(min, moda, max), erlang(k, escala), lognormal(mu, sigma).

PROCESSO (um gerador independente por terminal)
1. Notificação: intervalos exponenciais com a taxa do terminal. Sortear comprador ~ Bernoulli(min(1, prob_compra * fator_demanda))
   e, se comprador, o volume; senão volume 0.
2. Chegada ao fundeio após a antecedência (constante). 3. Espera FIFO por berço livre do terminal, sem desistência.
4. Manobra de entrada: o berço é ocupado no início dela. 5. Operação (janela de atendimento).
6. Manobra de saída, com o mesmo tempo da entrada; o berço é liberado ao fim. 7. Gravar o navio no log.
Variável de estado: N_j(t), navios da classe j no porto, da chegada ao fundeio até a saída.

SAÍDA: log_navios.csv, uma linha por navio: id_cenario, rep, id_navio, id_classe, id_terminal, t_notificacao, t_chegada,
t_ini_manobra, t_ini_operacao, t_fim_operacao, t_fim_manobra (data-hora a partir de data_inicio), comprador (0/1), volume_t.

EXECUÇÃO: uma replicação do cenário-base com horizonte de 365 dias; semente lida de cenario.csv.

VERIFICAÇÃO (semente fixa; para cada teste: objetivo, configuração, esperado, obtido, conclusão; executar de fato)
Cobrir geração de navios, notificação-chegada, capacidade dos berços, FIFO, manobras, operação, compradores, volume,
fator de demanda, coerência temporal e contrato do log, reprodutibilidade e parametrização. Usar horizontes curtos,
taxas alteradas, tempos determinísticos ou terminal único quando necessário.

ENTREGA: notebook na estrutura do modelo do professor, em português, executável do zero com caminhos relativos.
```

**Iterações subsequentes:** (1) comparação dos arquivos da atividade com o modelo conceitual; (2) geração de `terminais.csv` e `distancias.csv` a partir do arquivo de dados; (3) implementação e primeira execução, após a qual o diagnóstico passou a incluir os navios ainda em curso no horizonte; (4) escrita e execução da bateria de verificação; (5) após a revisão do modelo conceitual para a versão 1.1, inclusão dos campos acrescidos em `cenario.csv`, aplicação do fator de demanda e do teste V15; (6) execução do notebook em kernel limpo.

## 3. Modelo de simulação

O fluxo é *dados do cenário-base → arquivos de entrada → leitura dos parâmetros → simulação → log*. Nenhum parâmetro do cenário aparece no código: trocar de cenário exige apenas alterar os arquivos de `entrada/`.

### 3.1 Arquivos de entrada

`terminais.csv` e `distancias.csv` são gerados por funções que interpretam o arquivo de dados, para que os valores não sejam digitados à mão. As taxas são convertidas de navios/dia para navios/h e as manobras de h/min para horas; a matriz de distâncias é convertida para o formato longo. `cenario.csv` é editado diretamente, pois contém escolhas do cenário e não dados do sistema.

In [1]:
import re
from pathlib import Path

import pandas as pd

HORAS_POR_DIA = 24.0


def _num(txt):
    return float(txt.replace(",", "."))


def _secao(md, n):
    m = re.search(rf"^## {n}\.[^\n]*\n(.*?)(?=^## \d+\.|\Z)", md, flags=re.S | re.M)
    return m.group(1)


def _por_terminal(texto, padrao):
    return {m.group(1): m.groups()[1:]
            for m in re.finditer(rf"^- (T\d+): {padrao}\s*$", texto, flags=re.M)}


def _horas(txt):
    h = re.search(r"(\d+)\s*h", txt)
    mi = re.search(r"(\d+)\s*min", txt)
    return (int(h.group(1)) if h else 0) + (int(mi.group(1)) if mi else 0) / 60.0


def ler_dados_md(caminho_md):
    md = Path(caminho_md).read_text(encoding="utf-8")
    tri = r"Tri\(([\d,]+),\s*([\d,]+),\s*([\d,]+)\)"
    d = {
        "bercos": {k: int(v[0]) for k, v in _por_terminal(_secao(md, 1), r"(\d+) ber[çc]os?").items()},
        "taxa_dia": {k: _num(v[0]) for k, v in _por_terminal(_secao(md, 3), r"([\d,]+) navios?/dia").items()},
        "antecedencia": float(re.search(r"(\d+)\s*h", _secao(md, 5)).group(1)),
        "manobra": {k: _horas(v[0]) for k, v in _por_terminal(_secao(md, 7), r"(.+?)").items()},
        "operacao": {k: tuple(map(_num, v)) for k, v in _por_terminal(_secao(md, 8), tri + r"\s*h").items()},
        "prob": {k: _num(v[0]) / 100.0 for k, v in _por_terminal(_secao(md, 9), r"(\d+)%").items()},
        "volume": {k: tuple(map(_num, v)) for k, v in _por_terminal(_secao(md, 10), tri + r"\s*t").items()},
    }
    linhas = [l for l in _secao(md, 11).splitlines() if l.startswith("|")]
    destinos = [c.strip() for c in linhas[0].strip("|").split("|")][1:]
    dist = []
    for l in linhas[2:]:
        cel = [c.strip() for c in l.strip("|").split("|")]
        dist += [{"origem": cel[0], "destino": dest, "distancia": _num(v)} for dest, v in zip(destinos, cel[1:])]
    d["distancias"] = pd.DataFrame(dist)
    return d


def montar_terminais(d):
    linhas = []
    for t in sorted(d["bercos"], key=lambda x: int(x[1:])):
        op, vl = d["operacao"][t], d["volume"][t]
        linhas.append({
            "id_terminal": t, "id_classe": "C" + t[1:], "n_bercos": d["bercos"][t],
            "taxa_chegada": d["taxa_dia"][t] / HORAS_POR_DIA, "antecedencia": d["antecedencia"],
            "tempo_manobra": d["manobra"][t],
            "dist_operacao": "triangular", "op_p1": op[0], "op_p2": op[1], "op_p3": op[2],
            "prob_compra": d["prob"][t],
            "dist_volume": "triangular", "vl_p1": vl[0], "vl_p2": vl[1], "vl_p3": vl[2],
        })
    return pd.DataFrame(linhas)


def preparar_entradas(caminho_md, pasta_saida):
    pasta = Path(pasta_saida)
    pasta.mkdir(parents=True, exist_ok=True)
    d = ler_dados_md(caminho_md)
    terminais = montar_terminais(d)
    terminais.to_csv(pasta / "terminais.csv", index=False)
    d["distancias"].to_csv(pasta / "distancias.csv", index=False)
    return terminais, d["distancias"]

In [2]:
ARQ_DADOS = "entrada/D08_2_BVE_MComp_Navios_-_Dados.md"
preparar_entradas(ARQ_DADOS, "entrada")
display(pd.read_csv("entrada/terminais.csv"))
display(pd.read_csv("entrada/cenario.csv"))
display(pd.read_csv("entrada/distancias.csv").head(8))

,id_terminal,id_classe,n_bercos,taxa_chegada,antecedencia,tempo_manobra,dist_operacao,op_p1,op_p2,op_p3,prob_compra,dist_volume,vl_p1,vl_p2,vl_p3
0,T1,C1,3,0.104167,48.0,1.666667,triangular,12.0,15.0,19.0,0.70,triangular,600.0,1000.0,1800.0
1,T2,C2,5,0.166667,48.0,1.500000,triangular,14.0,18.0,22.0,0.60,triangular,800.0,1200.0,2000.0
2,T3,C3,4,0.187500,48.0,1.333333,triangular,9.0,12.0,15.0,0.50,triangular,400.0,700.0,1200.0
3,T4,C4,3,0.145833,48.0,1.333333,triangular,12.0,15.0,18.0,0.65,triangular,500.0,900.0,1600.0
4,T5,C5,2,0.083333,48.0,1.083333,triangular,9.0,12.0,15.0,0.55,triangular,300.0,600.0,1000.0
5,T6,C6,1,0.041667,48.0,0.916667,triangular,8.0,10.0,12.0,0.50,triangular,250.0,500.0,900.0


,id_cenario,data_inicio,horizonte,n_rep,semente,fator_demanda,horarios_programacao,horizonte_programacao,regra_pco
0,BASE,2026-01-01 00:00:00,365,1,20260,1.0,09:30;16:30,36,menor_fim_janela


,origem,destino,distancia
0,Base,Base,0.0
1,Base,T1,2.1
2,Base,T2,3.7
3,Base,T3,5.8
4,Base,T4,7.5
5,Base,T5,9.2
6,Base,T6,10.6
7,T1,Base,2.1


`distancias.csv` tem 49 linhas (7 × 7 locais, incluindo a base) e só será usado no submodelo das barcaças.

### 3.2 Modelo SimPy

Principais decisões de implementação:

- **Um gerador por terminal.** Cada terminal tem seu processo gerador de notificações (intervalos exponenciais com a taxa do terminal) e um `simpy.Resource` com capacidade `n_bercos`. O `Resource` atende as requisições na ordem de chegada, o que implementa a fila FIFO sem desistência.
- **Notificação.** Na notificação sorteiam-se o comprador, com probabilidade `min(1, prob_compra × fator_demanda)`, e o volume. O navio chega ao fundeio após a antecedência constante.
- **Berço.** O berço é requisitado na chegada ao fundeio e liberado ao fim da manobra de saída (bloco `with`); os instantes do log são gravados nos pontos correspondentes do processo.
- **Aleatoriedade reproduzível.** Para cada replicação, uma `SeedSequence([semente, rep])` gera fluxos independentes por terminal e por variável (chegadas, compra, volume, operação). Alterar berços ou *f_d* não altera os instantes de chegada, o que permite comparar cenários com números aleatórios comuns.
- **Estado.** O modelo acompanha *N_j(t)*, a ocupação máxima de berços e a fila máxima por terminal, usados na verificação. O relógio é numérico, em horas; a conversão para data-hora ocorre ao montar o log.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import simpy

HORAS_POR_DIA = 24.0

COLUNAS_LOG = ["id_cenario", "rep", "id_navio", "id_classe", "id_terminal",
               "t_notificacao", "t_chegada", "t_ini_manobra", "t_ini_operacao",
               "t_fim_operacao", "t_fim_manobra", "comprador", "volume_t"]
COLUNAS_TEMPO = [c for c in COLUNAS_LOG if c.startswith("t_")]


def carregar_parametros(pasta, id_cenario=None):
    pasta = Path(pasta)
    terminais = pd.read_csv(pasta / "terminais.csv")
    cenarios = pd.read_csv(pasta / "cenario.csv", parse_dates=["data_inicio"])
    if id_cenario is None:
        cenario = cenarios.iloc[0]
    else:
        cenario = cenarios.loc[cenarios["id_cenario"] == id_cenario].iloc[0]
    return terminais, cenario


def sortear(rng, dist, p1, p2=None, p3=None):
    if dist == "constante":
        return float(p1)
    if dist == "uniforme":
        return rng.uniform(p1, p2)
    if dist == "exponencial":
        return rng.exponential(p1)
    if dist == "triangular":
        return rng.triangular(p1, p2, p3)
    if dist == "erlang":
        return rng.gamma(p1, p2)
    if dist == "lognormal":
        return rng.lognormal(p1, p2)
    raise ValueError(f"Distribuição desconhecida: {dist}")


class Terminal:
    def __init__(self, env, linha, fator_demanda, sequencias):
        self.id = linha["id_terminal"]
        self.classe = linha["id_classe"]
        self.n_bercos = int(linha["n_bercos"])
        self.taxa = float(linha["taxa_chegada"])
        self.antecedencia = float(linha["antecedencia"])
        self.manobra = float(linha["tempo_manobra"])
        self.dist_op = linha["dist_operacao"]
        self.par_op = (linha["op_p1"], linha["op_p2"], linha["op_p3"])
        self.prob_compra = min(1.0, float(linha["prob_compra"]) * fator_demanda)
        self.dist_vol = linha["dist_volume"]
        self.par_vol = (linha["vl_p1"], linha["vl_p2"], linha["vl_p3"])
        self.bercos = simpy.Resource(env, capacity=self.n_bercos)
        self.rng_chegada, self.rng_compra, self.rng_volume, self.rng_operacao = (
            np.random.default_rng(s) for s in sequencias)
        self.N = 0
        self.N_max = 0
        self.n_notificados = 0
        self.ocupacao_max = 0
        self.fila_max = 0


def simular(terminais, cenario, rep):
    horizonte_h = float(cenario["horizonte"]) * HORAS_POR_DIA
    fator = float(cenario["fator_demanda"])
    env = simpy.Environment()
    n_fluxos = 4  # chegada, compra, volume, operação
    sementes = np.random.SeedSequence([int(cenario["semente"]), int(rep)]).spawn(n_fluxos * len(terminais))
    term = {}
    for i, (_, linha) in enumerate(terminais.iterrows()):
        term[linha["id_terminal"]] = Terminal(env, linha, fator, sementes[n_fluxos * i:n_fluxos * (i + 1)])

    registros, todos = [], []
    contador = {"navios": 0}

    def gerador_navios(t):
        while True:
            yield env.timeout(t.rng_chegada.exponential(1.0 / t.taxa))
            if env.now >= horizonte_h:
                break
            contador["navios"] += 1
            t.n_notificados += 1
            comprador = int(t.rng_compra.random() < t.prob_compra)
            volume = sortear(t.rng_volume, t.dist_vol, *t.par_vol) if comprador else 0.0
            reg = {"id_cenario": cenario["id_cenario"], "rep": int(rep),
                   "id_navio": f"N{contador['navios']:06d}", "id_classe": t.classe,
                   "id_terminal": t.id, "t_notificacao": env.now, "t_chegada": np.nan,
                   "t_ini_manobra": np.nan, "t_ini_operacao": np.nan, "t_fim_operacao": np.nan,
                   "t_fim_manobra": np.nan, "comprador": comprador, "volume_t": volume}
            todos.append(reg)
            env.process(navio(t, reg))

    def navio(t, reg):
        yield env.timeout(t.antecedencia)
        reg["t_chegada"] = env.now
        t.N += 1
        t.N_max = max(t.N_max, t.N)
        with t.bercos.request() as req:
            t.fila_max = max(t.fila_max, len(t.bercos.queue))
            yield req
            reg["t_ini_manobra"] = env.now
            t.ocupacao_max = max(t.ocupacao_max, t.bercos.count)
            yield env.timeout(t.manobra)
            reg["t_ini_operacao"] = env.now
            yield env.timeout(sortear(t.rng_operacao, t.dist_op, *t.par_op))
            reg["t_fim_operacao"] = env.now
            yield env.timeout(t.manobra)
            reg["t_fim_manobra"] = env.now
        t.N -= 1
        registros.append(reg)

    for t in term.values():
        env.process(gerador_navios(t))
    env.run(until=horizonte_h)

    diag = {
        "horizonte_h": horizonte_h,
        "n_notificados": {k: t.n_notificados for k, t in term.items()},
        "ocupacao_max": {k: t.ocupacao_max for k, t in term.items()},
        "fila_max": {k: t.fila_max for k, t in term.items()},
        "N_max": {k: t.N_max for k, t in term.items()},
        "N_final": {k: t.N for k, t in term.items()},
        "n_registrados": len(registros),
        "n_total_notificados": contador["navios"],
        "notificados": montar_log(todos, cenario),
    }
    return montar_log(registros, cenario), diag


def montar_log(registros, cenario):
    log = pd.DataFrame(registros, columns=COLUNAS_LOG)
    origem = pd.Timestamp(cenario["data_inicio"])
    for c in COLUNAS_TEMPO:
        log[c] = origem + pd.to_timedelta(log[c], unit="h")
    log = log.sort_values("id_navio").reset_index(drop=True)
    log["rep"] = log["rep"].astype(int)
    log["comprador"] = log["comprador"].astype(int)
    log["volume_t"] = log["volume_t"].astype(float)
    return log


def gravar_log(log, caminho):
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    log.to_csv(caminho, index=False, date_format="%Y-%m-%d %H:%M:%S.%f")


def executar(pasta_entrada, pasta_saida, id_cenario=None):
    terminais, cenario = carregar_parametros(pasta_entrada, id_cenario)
    logs, diags = [], []
    for rep in range(1, int(cenario["n_rep"]) + 1):
        log, diag = simular(terminais, cenario, rep)
        logs.append(log)
        diags.append(diag)
    log = pd.concat(logs, ignore_index=True)
    gravar_log(log, Path(pasta_saida) / "log_navios.csv")
    return log, diags

## 4. Saída

Replicação do cenário-base (365 dias), lida dos arquivos de entrada e gravada em `saida/log_navios.csv`.

In [4]:
terminais, cenario = carregar_parametros("entrada")
print("Cenário:", cenario["id_cenario"], "| horizonte (dias):", cenario["horizonte"],
      "| replicações:", cenario["n_rep"], "| semente:", cenario["semente"],
      "| fator de demanda:", cenario["fator_demanda"])

log_base, diagnosticos = executar("entrada", "saida")
diag_base = diagnosticos[0]

print("Navios notificados no horizonte:", diag_base["n_total_notificados"])
print("Navios no log (saída concluída até o horizonte):", len(log_base))
print("Navios ainda em curso no fim do horizonte:", diag_base["n_total_notificados"] - len(log_base))
display(log_base.head(8))

Cenário: BASE | horizonte (dias): 365 | replicações: 1 | semente: 20260 | fator de demanda: 1.0
Navios notificados no horizonte: 6467
Navios no log (saída concluída até o horizonte): 6407
Navios ainda em curso no fim do horizonte: 60


,id_cenario,rep,id_navio,id_classe,id_terminal,t_notificacao,t_chegada,t_ini_manobra,t_ini_operacao,t_fim_operacao,t_fim_manobra,comprador,volume_t
0,BASE,1,N000001,C2,T2,2026-01-01 00:43:57.791823859,2026-01-03 00:43:57.791823859,2026-01-03 00:43:57.791823859,2026-01-03 02:13:57.791823859,2026-01-03 16:34:25.207759959,2026-01-03 18:04:25.207759959,0,0.000000
1,BASE,1,N000002,C1,T1,2026-01-01 02:32:36.503655273,2026-01-03 02:32:36.503655273,2026-01-03 02:32:36.503655273,2026-01-03 04:12:36.503655274,2026-01-03 19:39:02.141721997,2026-01-03 21:19:02.141721994,1,1416.468533
2,BASE,1,N000003,C5,T5,2026-01-01 04:48:15.535278698,2026-01-03 04:48:15.535278698,2026-01-03 04:48:15.535278698,2026-01-03 05:53:15.535278697,2026-01-03 19:04:16.391714739,2026-01-03 20:09:16.391714738,0,0.000000
3,BASE,1,N000004,C4,T4,2026-01-01 07:20:52.931873248,2026-01-03 07:20:52.931873248,2026-01-03 07:20:52.931873248,2026-01-03 08:40:52.931873247,2026-01-03 23:07:18.563062317,2026-01-04 00:27:18.563062316,0,0.000000
4,BASE,1,N000005,C4,T4,2026-01-01 11:53:19.625942569,2026-01-03 11:53:19.625942569,2026-01-03 11:53:19.625942569,2026-01-03 13:13:19.625942571,2026-01-04 03:39:27.440770597,2026-01-04 04:59:27.440770599,1,823.652010
5,BASE,1,N000006,C3,T3,2026-01-01 13:27:40.901692982,2026-01-03 13:27:40.901692982,2026-01-03 13:27:40.901692982,2026-01-03 14:47:40.901692984,2026-01-04 04:32:16.680557209,2026-01-04 05:52:16.680557208,1,676.169772
6,BASE,1,N000007,C1,T1,2026-01-01 14:44:12.863689339,2026-01-03 14:44:12.863689339,2026-01-03 14:44:12.863689339,2026-01-03 16:24:12.863689340,2026-01-04 06:42:12.513803353,2026-01-04 08:22:12.513803354,1,1082.321575
7,BASE,1,N000008,C6,T6,2026-01-01 15:26:02.504099475,2026-01-03 15:26:02.504099475,2026-01-03 15:26:02.504099475,2026-01-03 16:21:02.504099473,2026-01-04 03:00:31.585920706,2026-01-04 03:55:31.585920704,0,0.000000


In [5]:
def _h(s):
    return s.dt.total_seconds() / 3600

H_base = float(cenario["horizonte"]) * HORAS_POR_DIA
resumo = (log_base.assign(espera_h=_h(log_base["t_ini_manobra"] - log_base["t_chegada"]),
                          permanencia_h=_h(log_base["t_fim_manobra"] - log_base["t_chegada"]),
                          janela_h=_h(log_base["t_fim_operacao"] - log_base["t_ini_operacao"]),
                          ocupacao_h=_h(log_base["t_fim_manobra"] - log_base["t_ini_manobra"]))
          .groupby("id_terminal")
          .agg(navios=("id_navio", "count"), pedidos=("comprador", "sum"), volume_t=("volume_t", "sum"),
               espera_media_h=("espera_h", "mean"), permanencia_media_h=("permanencia_h", "mean"),
               janela_media_h=("janela_h", "mean"), ocupacao_h=("ocupacao_h", "sum")))
resumo["utilizacao"] = resumo["ocupacao_h"] / (terminais.set_index("id_terminal")["n_bercos"] * H_base)
display(resumo.drop(columns="ocupacao_h").round(2))

,navios,pedidos,volume_t,espera_media_h,permanencia_media_h,janela_media_h,utilizacao
id_terminal,,,,,,,
T1,881,603,689397.12,3.40,21.98,15.25,0.62
T2,1490,913,1213450.36,3.46,24.50,18.04,0.72
T3,1675,834,629848.04,2.85,17.52,12.00,0.70
T4,1285,878,878821.94,12.96,30.66,15.03,0.87
T5,696,372,237153.47,3.42,17.65,12.07,0.57
T6,380,181,98403.48,5.75,17.58,10.00,0.51


O trecho acima mostra os primeiros navios do log; a tabela resume os indicadores dos navios previstos no modelo conceitual (seção 6), calculados a partir do log. T4 tem a maior utilização (cerca de 0,86) e, por isso, a maior espera no fundeio.

**Campos do `log_navios.csv`** (uma linha por navio, registro bruto):

| Campo | Descrição |
|:--|:--|
| `id_cenario`, `rep` | Cenário e replicação |
| `id_navio` | Identificador único do navio (ordem de notificação) |
| `id_classe`, `id_terminal` | Classe do navio e terminal de atracação |
| `t_notificacao` | Notificação da chegada; nesse instante são definidos `comprador` e `volume_t` |
| `t_chegada` | Chegada ao fundeio (notificação + antecedência) |
| `t_ini_manobra` | Início da manobra de entrada; o berço passa a estar ocupado |
| `t_ini_operacao` | Início da operação; abre a janela de atendimento |
| `t_fim_operacao` | Fim da operação; fecha a janela |
| `t_fim_manobra` | Fim da manobra de saída; libera o berço |
| `comprador` | 1 se o navio demanda combustível, 0 caso contrário |
| `volume_t` | Volume demandado (t); zero se `comprador` = 0 |

Os instantes estão em data-hora a partir de `data_inicio`. No submodelo 3, cada linha com `comprador` = 1 é um pedido: `t_notificacao` é a entrada na carteira do PCO, `id_terminal` o local de atendimento, [`t_ini_operacao`, `t_fim_operacao`] a janela e `volume_t` a quantidade.

## 5. Verificação do modelo

Todos os testes usam a semente de `cenario.csv`. Os cenários de teste alteram em memória as tabelas lidas dos arquivos (horizontes curtos, taxas alteradas, tempos determinísticos, terminal único), sem mudar a lógica do modelo. Os critérios estatísticos usam tolerância de 3 desvios-padrão. Cada teste registra objetivo, configuração, resultado esperado, resultado obtido e conclusão; a tabela final resume os resultados.

In [6]:
from IPython.display import Markdown

terminais_base, cenario_base = carregar_parametros("entrada")
origem = pd.Timestamp(cenario_base["data_inicio"])
H30 = 30
RESULTADOS = []


def cen(**kw):
    c = cenario_base.copy()
    for k, v in kw.items():
        c[k] = v
    return c


def horas(serie):
    return serie.dt.total_seconds() / 3600


def relativo(serie):
    return horas(serie - origem)


def relatar(codigo, nome, objetivo, config, esperado, obtido, ok, conclusao):
    ok = bool(ok)
    RESULTADOS.append({"Teste": codigo, "Descrição": nome, "Resultado": "Aprovado" if ok else "Reprovado"})
    display(Markdown(
        f"**{codigo} – {nome}**\n\n"
        f"- **Objetivo:** {objetivo}\n"
        f"- **Configuração:** {config}\n"
        f"- **Esperado:** {esperado}\n"
        f"- **Obtido:** {obtido}\n"
        f"- **Conclusão:** {'Aprovado. ' if ok else 'REPROVADO. '}{conclusao}"))


def ocupacao_maxima(log, terminal):
    d = log[log["id_terminal"] == terminal]
    ev = sorted([(t, 1) for t in d["t_ini_manobra"]] + [(t, -1) for t in d["t_fim_manobra"]],
                key=lambda x: (x[0], x[1]))
    atual = maximo = 0
    for _, delta in ev:
        atual += delta
        maximo = max(maximo, atual)
    return maximo


def esperas(log):
    return horas(log["t_ini_manobra"] - log["t_chegada"])

In [7]:
# V01 - Arquivos de entrada
esperado_df = montar_terminais(ler_dados_md(ARQ_DADOS))
lido = pd.read_csv("entrada/terminais.csv")
try:
    pd.testing.assert_frame_equal(lido, esperado_df, check_exact=False, atol=1e-12)
    iguais = True
except AssertionError:
    iguais = False
n_dist = len(pd.read_csv("entrada/distancias.csv"))
dist_ok = n_dist == (len(terminais_base) + 1) ** 2
campos_cen = ["id_cenario", "data_inicio", "horizonte", "n_rep", "semente",
              "fator_demanda", "horarios_programacao", "horizonte_programacao", "regra_pco"]
cen_ok = list(pd.read_csv("entrada/cenario.csv").columns) == campos_cen
relatar("V01", "Arquivos de entrada",
        "Confirmar que os arquivos de entrada reproduzem os dados do cenário-base e seguem o formato do modelo conceitual.",
        "`terminais.csv` relido e comparado com nova leitura do arquivo de dados; linhas de `distancias.csv`; colunas e horizonte de `cenario.csv`.",
        "`terminais.csv` idêntico à leitura do arquivo de dados; `distancias.csv` com (M+1)² linhas; `cenario.csv` com os 9 campos do modelo conceitual e horizonte de 365 dias.",
        f"terminais idênticos: {iguais}; linhas em `distancias.csv`: {n_dist} (M = {len(terminais_base)}); campos de `cenario.csv` conforme: {cen_ok}; horizonte: {cenario_base['horizonte']} dias.",
        iguais and dist_ok and cen_ok and float(cenario_base["horizonte"]) == 365,
        "Os parâmetros do cenário-base vêm do arquivo de dados e de `cenario.csv`, no formato definido pelo modelo conceitual.")

**V01 – Arquivos de entrada**

- **Objetivo:** Confirmar que os arquivos de entrada reproduzem os dados do cenário-base e seguem o formato do modelo conceitual.
- **Configuração:** `terminais.csv` relido e comparado com nova leitura do arquivo de dados; linhas de `distancias.csv`; colunas e horizonte de `cenario.csv`.
- **Esperado:** `terminais.csv` idêntico à leitura do arquivo de dados; `distancias.csv` com (M+1)² linhas; `cenario.csv` com os 9 campos do modelo conceitual e horizonte de 365 dias.
- **Obtido:** terminais idênticos: True; linhas em `distancias.csv`: 49 (M = 6); campos de `cenario.csv` conforme: True; horizonte: 365 dias.
- **Conclusão:** Aprovado. Os parâmetros do cenário-base vêm do arquivo de dados e de `cenario.csv`, no formato definido pelo modelo conceitual.

In [8]:
# V02 - Geração de navios (Poisson por terminal)
_, d30 = simular(terminais_base, cen(horizonte=H30), 1)
linhas, ok_cont = [], True
for _, r in terminais_base.iterrows():
    esp = r["taxa_chegada"] * H30 * HORAS_POR_DIA
    n = d30["n_notificados"][r["id_terminal"]]
    ok_cont &= abs(n - esp) <= 3 * np.sqrt(esp)
    linhas.append(f"{r['id_terminal']}: {n} (esperado {esp:.1f} ± {3*np.sqrt(esp):.1f})")
ok_int, linhas_int = True, []
notif = diag_base["notificados"]
for _, r in terminais_base.iterrows():
    x = horas(notif.loc[notif["id_terminal"] == r["id_terminal"], "t_notificacao"].sort_values().diff().dropna())
    media_esp = 1 / r["taxa_chegada"]
    ok_int &= abs(x.mean() - media_esp) <= 3 * x.std() / np.sqrt(len(x)) and abs(x.std() / x.mean() - 1) <= 0.2
    linhas_int.append(f"{r['id_terminal']}: média {x.mean():.2f} h (esperado {media_esp:.2f}), CV {x.std()/x.mean():.2f}")
relatar("V02", "Geração de navios",
        "Verificar que cada terminal gera notificações como processo de Poisson com a taxa lida de `terminais.csv`.",
        f"(a) Contagem de notificações por terminal em {H30} dias; (b) intervalos entre notificações no cenário-base.",
        "(a) Contagem dentro de λ·T ± 3√(λ·T); (b) média dos intervalos igual a 1/λ (3 erros-padrão) e coeficiente de variação próximo de 1 (|CV − 1| ≤ 0,2).",
        "(a) " + "; ".join(linhas) + ". (b) " + "; ".join(linhas_int) + ".",
        ok_cont and ok_int,
        "O número e o espaçamento das notificações são compatíveis com um processo de Poisson de taxa λ_j.")

**V02 – Geração de navios**

- **Objetivo:** Verificar que cada terminal gera notificações como processo de Poisson com a taxa lida de `terminais.csv`.
- **Configuração:** (a) Contagem de notificações por terminal em 30 dias; (b) intervalos entre notificações no cenário-base.
- **Esperado:** (a) Contagem dentro de λ·T ± 3√(λ·T); (b) média dos intervalos igual a 1/λ (3 erros-padrão) e coeficiente de variação próximo de 1 (|CV − 1| ≤ 0,2).
- **Obtido:** (a) T1: 81 (esperado 75.0 ± 26.0); T2: 124 (esperado 120.0 ± 32.9); T3: 145 (esperado 135.0 ± 34.9); T4: 109 (esperado 105.0 ± 30.7); T5: 51 (esperado 60.0 ± 23.2); T6: 27 (esperado 30.0 ± 16.4). (b) T1: média 9.87 h (esperado 9.60), CV 1.01; T2: média 5.83 h (esperado 6.00), CV 0.98; T3: média 5.18 h (esperado 5.33), CV 1.00; T4: média 6.73 h (esperado 6.86), CV 1.03; T5: média 12.46 h (esperado 12.00), CV 1.02; T6: média 22.77 h (esperado 24.00), CV 0.96.
- **Conclusão:** Aprovado. O número e o espaçamento das notificações são compatíveis com um processo de Poisson de taxa λ_j.

In [9]:
# V03 - Notificação -> chegada
t_ant = terminais_base.copy()
t_ant["antecedencia"] = t_ant["antecedencia"] * 0.5
_, dA = simular(t_ant, cen(horizonte=H30), 1)
nt = dA["notificados"].merge(t_ant[["id_terminal", "antecedencia"]], on="id_terminal")
chegou = nt["t_chegada"].notna()
erro = (horas(nt.loc[chegou, "t_chegada"] - nt.loc[chegou, "t_notificacao"]) - nt.loc[chegou, "antecedencia"]).abs().max()
pend = nt.loc[~chegou]
pend_ok = ((relativo(pend["t_notificacao"]) + pend["antecedencia"]) >= H30 * HORAS_POR_DIA - 1e-9).all()
relatar("V03", "Relação notificação → chegada",
        "Confirmar que a chegada ao fundeio ocorre `antecedencia` horas após a notificação, com o valor lido do arquivo.",
        f"Antecedência reduzida à metade do valor do arquivo; horizonte de {H30} dias; todos os navios notificados.",
        "`t_chegada − t_notificacao` igual à antecedência (erro ≤ 1e-8 h); só não chegam os navios cuja chegada cairia após o horizonte.",
        f"Erro máximo: {erro:.2e} h em {int(chegou.sum())} navios; navios sem chegada: {len(pend)} (todos com chegada prevista após o horizonte: {bool(pend_ok)}).",
        erro <= 1e-8 and pend_ok,
        "A antecedência é uma constante lida do arquivo; alterar o arquivo muda o comportamento sem alterar o código.")

**V03 – Relação notificação → chegada**

- **Objetivo:** Confirmar que a chegada ao fundeio ocorre `antecedencia` horas após a notificação, com o valor lido do arquivo.
- **Configuração:** Antecedência reduzida à metade do valor do arquivo; horizonte de 30 dias; todos os navios notificados.
- **Esperado:** `t_chegada − t_notificacao` igual à antecedência (erro ≤ 1e-8 h); só não chegam os navios cuja chegada cairia após o horizonte.
- **Obtido:** Erro máximo: 8.31e-13 h em 512 navios; navios sem chegada: 25 (todos com chegada prevista após o horizonte: True).
- **Conclusão:** Aprovado. A antecedência é uma constante lida do arquivo; alterar o arquivo muda o comportamento sem alterar o código.

In [10]:
# V04 - Capacidade dos berços
t_sat = terminais_base.copy()
t_sat["taxa_chegada"] = t_sat["taxa_chegada"] * 3
log_sat, d_sat = simular(t_sat, cen(horizonte=H30), 1)
res, ok_cap = [], True
for _, r in terminais_base.iterrows():
    j, c = r["id_terminal"], int(r["n_bercos"])
    m_base, m_sat = ocupacao_maxima(log_base, j), ocupacao_maxima(log_sat, j)
    ok_cap &= (m_base <= c) and (m_sat == c) and d_sat["ocupacao_max"][j] == c and d_sat["fila_max"][j] > 0
    res.append(f"{j}: c = {c}, máx. base = {m_base}, máx. carga triplicada = {m_sat} (fila máx. {d_sat['fila_max'][j]})")
relatar("V04", "Capacidade dos berços",
        "Verificar que nunca há mais navios ocupando berço do que `n_bercos` e que o limite é atingido sob carga alta.",
        f"Ocupação calculada a partir do log (`t_ini_manobra` a `t_fim_manobra`) no cenário-base e com taxas triplicadas ({H30} dias).",
        "Ocupação máxima ≤ c_j no cenário-base; = c_j sob carga alta, com fila; monitor interno igual ao log.",
        "; ".join(res) + ".",
        ok_cap,
        "A capacidade de cada terminal é respeitada e saturada quando a demanda excede os berços.")

**V04 – Capacidade dos berços**

- **Objetivo:** Verificar que nunca há mais navios ocupando berço do que `n_bercos` e que o limite é atingido sob carga alta.
- **Configuração:** Ocupação calculada a partir do log (`t_ini_manobra` a `t_fim_manobra`) no cenário-base e com taxas triplicadas (30 dias).
- **Esperado:** Ocupação máxima ≤ c_j no cenário-base; = c_j sob carga alta, com fila; monitor interno igual ao log.
- **Obtido:** T1: c = 3, máx. base = 3, máx. carga triplicada = 3 (fila máx. 98); T2: c = 5, máx. base = 5, máx. carga triplicada = 5 (fila máx. 171); T3: c = 4, máx. base = 4, máx. carga triplicada = 4 (fila máx. 206); T4: c = 3, máx. base = 3, máx. carga triplicada = 3 (fila máx. 187); T5: c = 2, máx. base = 2, máx. carga triplicada = 2 (fila máx. 56); T6: c = 1, máx. base = 1, máx. carga triplicada = 1 (fila máx. 21).
- **Conclusão:** Aprovado. A capacidade de cada terminal é respeitada e saturada quando a demanda excede os berços.

In [11]:
# V05 - FIFO
ok_fifo, res = True, []
for nome, lg in (("base", log_base), ("carga triplicada", log_sat)):
    for j, d in lg.groupby("id_terminal"):
        d = d.sort_values("t_chegada")
        mono = (d["t_ini_manobra"].diff().dropna() >= pd.Timedelta(0)).all()
        mesmo = (d["t_chegada"].rank(method="first").values == d["t_ini_manobra"].rank(method="first").values).all()
        ok_fifo &= bool(mono and mesmo)
    res.append(f"{nome}: {int((esperas(lg) > 1e-9).sum())} navios esperaram no fundeio")
relatar("V05", "Disciplina FIFO",
        "Confirmar que, em cada terminal, os navios iniciam a manobra de entrada na ordem de chegada ao fundeio.",
        "Ordem de `t_chegada` comparada com a de `t_ini_manobra` por terminal, no cenário-base e com carga triplicada.",
        "Nenhuma ultrapassagem em nenhum terminal.",
        f"Ultrapassagens: {'nenhuma' if ok_fifo else 'encontradas'}; " + "; ".join(res) + ".",
        ok_fifo,
        "A fila de cada terminal é FIFO, sem prioridade nem desistência, inclusive com espera efetiva.")

**V05 – Disciplina FIFO**

- **Objetivo:** Confirmar que, em cada terminal, os navios iniciam a manobra de entrada na ordem de chegada ao fundeio.
- **Configuração:** Ordem de `t_chegada` comparada com a de `t_ini_manobra` por terminal, no cenário-base e com carga triplicada.
- **Esperado:** Nenhuma ultrapassagem em nenhum terminal.
- **Obtido:** Ultrapassagens: nenhuma; base: 3030 navios esperaram no fundeio; carga triplicada: 681 navios esperaram no fundeio.
- **Conclusão:** Aprovado. A fila de cada terminal é FIFO, sem prioridade nem desistência, inclusive com espera efetiva.

In [12]:
# V06 - Manobra e ocupação do berço (terminal único com 1 berço, tempos determinísticos)
t1 = terminais_base[terminais_base["n_bercos"] == 1].copy()
t1["dist_operacao"] = "constante"
t1["op_p1"] = t1["op_p2"]
t1["taxa_chegada"] = t1["taxa_chegada"] * 3
log1, _ = simular(t1, cen(horizonte=20), 1)
m, op = float(t1["tempo_manobra"].iloc[0]), float(t1["op_p1"].iloc[0])
d = log1.sort_values("t_chegada")
cheg, ini = relativo(d["t_chegada"]).values, relativo(d["t_ini_manobra"]).values
esp_ini, esp_alt, prev, prev_alt = [], [], -np.inf, -np.inf
for c in cheg:
    i1 = max(c, prev); prev = i1 + 2 * m + op
    i2 = max(c, prev_alt); prev_alt = i2 + m + op
    esp_ini.append(i1); esp_alt.append(i2)
erro_ini = np.abs(np.array(esp_ini) - ini).max()
dif_alt = int((np.abs(np.array(esp_alt) - ini) > 1e-6).sum())
man = log_base["id_terminal"].map(terminais_base.set_index("id_terminal")["tempo_manobra"])
err_ent = (horas(log_base["t_ini_operacao"] - log_base["t_ini_manobra"]) - man).abs().max()
err_sai = (horas(log_base["t_fim_manobra"] - log_base["t_fim_operacao"]) - man).abs().max()
err_dur = (horas(d["t_fim_manobra"] - d["t_ini_manobra"]) - (2 * m + op)).abs().max()
relatar("V06", "Manobras e ocupação do berço",
        "Verificar os tempos de manobra e que o berço fica ocupado da manobra de entrada ao fim da manobra de saída.",
        f"(a) Todos os navios do cenário-base. (b) Terminal de 1 berço, operação constante ({op:g} h), taxa triplicada, 20 dias; início esperado pela recursão início_k = max(chegada_k, fim_(k−1)).",
        "(a) Manobras iguais a `tempo_manobra` (erro ≤ 1e-8 h). (b) Início igual ao da recursão (erro ≤ 1e-6 h); ocupação = 2·manobra + operação; a hipótese alternativa (berço liberado no fim da operação) daria inícios diferentes.",
        f"(a) erro máximo na entrada {err_ent:.1e} h e na saída {err_sai:.1e} h ({len(log_base)} navios). (b) {len(d)} navios; erro do início {erro_ini:.1e} h; erro da ocupação {err_dur:.1e} h; a hipótese alternativa diferiria em {dif_alt} navios.",
        err_ent <= 1e-8 and err_sai <= 1e-8 and erro_ini <= 1e-6 and err_dur <= 1e-6 and dif_alt > 0,
        "O berço fica ocupado durante as duas manobras e a operação; os tempos de manobra seguem o arquivo.")

**V06 – Manobras e ocupação do berço**

- **Objetivo:** Verificar os tempos de manobra e que o berço fica ocupado da manobra de entrada ao fim da manobra de saída.
- **Configuração:** (a) Todos os navios do cenário-base. (b) Terminal de 1 berço, operação constante (10 h), taxa triplicada, 20 dias; início esperado pela recursão início_k = max(chegada_k, fim_(k−1)).
- **Esperado:** (a) Manobras iguais a `tempo_manobra` (erro ≤ 1e-8 h). (b) Início igual ao da recursão (erro ≤ 1e-6 h); ocupação = 2·manobra + operação; a hipótese alternativa (berço liberado no fim da operação) daria inícios diferentes.
- **Obtido:** (a) erro máximo na entrada 8.3e-13 h e na saída 8.3e-13 h (6407 navios). (b) 36 navios; erro do início 9.1e-13 h; erro da ocupação 8.3e-13 h; a hipótese alternativa diferiria em 35 navios.
- **Conclusão:** Aprovado. O berço fica ocupado durante as duas manobras e a operação; os tempos de manobra seguem o arquivo.

In [13]:
# V07 - Tempo de operação
ok_op, res = True, []
for _, r in terminais_base.iterrows():
    x = horas((d_ := log_base[log_base["id_terminal"] == r["id_terminal"]])["t_fim_operacao"] - d_["t_ini_operacao"])
    a, b, c = r["op_p1"], r["op_p2"], r["op_p3"]
    media = (a + b + c) / 3
    ok_op &= x.min() >= a - 1e-9 and x.max() <= c + 1e-9 and abs(x.mean() - media) <= 3 * x.std() / np.sqrt(len(x))
    res.append(f"{r['id_terminal']}: [{x.min():.2f}; {x.max():.2f}] h, média {x.mean():.2f} (esperada {media:.2f})")
err_const = (horas(log1["t_fim_operacao"] - log1["t_ini_operacao"]) - op).abs().max()
relatar("V07", "Operação no terminal",
        "Verificar que a duração da operação (janela de atendimento) segue a distribuição triangular do terminal.",
        "Duração `t_fim_operacao − t_ini_operacao` por terminal no cenário-base e no caso determinístico de V06.",
        "Valores entre mínimo e máximo; média igual a (mín + moda + máx)/3 (3 erros-padrão); no caso constante, duração igual ao parâmetro.",
        "; ".join(res) + f". Caso constante: erro máximo {err_const:.1e} h.",
        ok_op and err_const <= 1e-8,
        "As durações de operação respeitam a distribuição e os parâmetros do arquivo.")

**V07 – Operação no terminal**

- **Objetivo:** Verificar que a duração da operação (janela de atendimento) segue a distribuição triangular do terminal.
- **Configuração:** Duração `t_fim_operacao − t_ini_operacao` por terminal no cenário-base e no caso determinístico de V06.
- **Esperado:** Valores entre mínimo e máximo; média igual a (mín + moda + máx)/3 (3 erros-padrão); no caso constante, duração igual ao parâmetro.
- **Obtido:** T1: [12.15; 18.87] h, média 15.25 (esperada 15.33); T2: [14.13; 21.93] h, média 18.04 (esperada 18.00); T3: [9.07; 14.94] h, média 12.00 (esperada 12.00); T4: [12.10; 17.90] h, média 15.03 (esperada 15.00); T5: [9.15; 14.87] h, média 12.07 (esperada 12.00); T6: [8.19; 11.87] h, média 10.00 (esperada 10.00). Caso constante: erro máximo 0.0e+00 h.
- **Conclusão:** Aprovado. As durações de operação respeitam a distribuição e os parâmetros do arquivo.

In [14]:
# V08 - Compradores
ok_cmp, res = True, []
for _, r in terminais_base.iterrows():
    d_ = log_base[log_base["id_terminal"] == r["id_terminal"]]
    p, n = r["prob_compra"], len(d_)
    frac = d_["comprador"].mean()
    ok_cmp &= abs(frac - p) <= 3 * np.sqrt(p * (1 - p) / n)
    res.append(f"{r['id_terminal']}: {frac:.3f} (p = {p:.2f}, n = {n})")
t_p0, t_p1 = terminais_base.copy(), terminais_base.copy()
t_p0["prob_compra"], t_p1["prob_compra"] = 0.0, 1.0
_, dp0 = simular(t_p0, cen(horizonte=H30), 1)
_, dp1 = simular(t_p1, cen(horizonte=H30), 1)
ext_ok = dp0["notificados"]["comprador"].sum() == 0 and dp1["notificados"]["comprador"].all()
relatar("V08", "Geração de compradores",
        "Verificar que a condição de comprador é sorteada com a probabilidade `prob_compra` do terminal.",
        f"Fração de compradores por terminal no cenário-base; casos extremos `prob_compra` = 0 e = 1 ({H30} dias).",
        "Fração dentro de p ± 3√(p(1−p)/n); nenhum comprador com p = 0; todos com p = 1.",
        "; ".join(res) + f". Extremos: p = 0 → {int(dp0['notificados']['comprador'].sum())} de {len(dp0['notificados'])}; p = 1 → {int(dp1['notificados']['comprador'].sum())} de {len(dp1['notificados'])}.",
        ok_cmp and ext_ok,
        "A condição de comprador segue a Bernoulli parametrizada por terminal.")

**V08 – Geração de compradores**

- **Objetivo:** Verificar que a condição de comprador é sorteada com a probabilidade `prob_compra` do terminal.
- **Configuração:** Fração de compradores por terminal no cenário-base; casos extremos `prob_compra` = 0 e = 1 (30 dias).
- **Esperado:** Fração dentro de p ± 3√(p(1−p)/n); nenhum comprador com p = 0; todos com p = 1.
- **Obtido:** T1: 0.684 (p = 0.70, n = 881); T2: 0.613 (p = 0.60, n = 1490); T3: 0.498 (p = 0.50, n = 1675); T4: 0.683 (p = 0.65, n = 1285); T5: 0.534 (p = 0.55, n = 696); T6: 0.476 (p = 0.50, n = 380). Extremos: p = 0 → 0 de 537; p = 1 → 537 de 537.
- **Conclusão:** Aprovado. A condição de comprador segue a Bernoulli parametrizada por terminal.

In [15]:
# V09 - Volume demandado
ok_vol, res = True, []
for _, r in terminais_base.iterrows():
    x = log_base[(log_base["id_terminal"] == r["id_terminal"]) & (log_base["comprador"] == 1)]["volume_t"]
    a, b, c = r["vl_p1"], r["vl_p2"], r["vl_p3"]
    media = (a + b + c) / 3
    ok_vol &= x.min() >= a and x.max() <= c and abs(x.mean() - media) <= 3 * x.std() / np.sqrt(len(x))
    res.append(f"{r['id_terminal']}: [{x.min():.0f}; {x.max():.0f}] t, média {x.mean():.1f} (esperada {media:.1f})")
coer = ((log_base["comprador"] == 0) == (log_base["volume_t"] == 0)).all()
antes = diag_base["notificados"][diag_base["notificados"]["t_chegada"].isna()]
define_not = len(antes) > 0 and ((antes["comprador"] == 1) == (antes["volume_t"] > 0)).all()
relatar("V09", "Geração da demanda (volume)",
        "Verificar o volume demandado (triangular por terminal) e que a demanda é definida na notificação.",
        "Volumes dos compradores do cenário-base; coerência entre `comprador` e `volume_t`; navios notificados que ainda não chegaram no fim do horizonte.",
        "Volumes entre mínimo e máximo, média igual a (mín + moda + máx)/3 (3 erros-padrão); `volume_t` = 0 se e só se `comprador` = 0; navios só notificados já têm a demanda definida.",
        "; ".join(res) + f". Coerência comprador/volume: {bool(coer)}. Navios só notificados: {len(antes)}, todos com demanda definida: {bool(define_not)}.",
        ok_vol and coer and define_not,
        "A demanda é sorteada na notificação, com os parâmetros do arquivo, e registrada para uso pelo PCO.")

**V09 – Geração da demanda (volume)**

- **Objetivo:** Verificar o volume demandado (triangular por terminal) e que a demanda é definida na notificação.
- **Configuração:** Volumes dos compradores do cenário-base; coerência entre `comprador` e `volume_t`; navios notificados que ainda não chegaram no fim do horizonte.
- **Esperado:** Volumes entre mínimo e máximo, média igual a (mín + moda + máx)/3 (3 erros-padrão); `volume_t` = 0 se e só se `comprador` = 0; navios só notificados já têm a demanda definida.
- **Obtido:** T1: [637; 1777] t, média 1143.3 (esperada 1133.3); T2: [841; 1969] t, média 1329.1 (esperada 1333.3); T3: [401; 1178] t, média 755.2 (esperada 766.7); T4: [505; 1579] t, média 1000.9 (esperada 1000.0); T5: [304; 991] t, média 637.5 (esperada 633.3); T6: [285; 875] t, média 543.7 (esperada 550.0). Coerência comprador/volume: True. Navios só notificados: 37, todos com demanda definida: True.
- **Conclusão:** Aprovado. A demanda é sorteada na notificação, com os parâmetros do arquivo, e registrada para uso pelo PCO.

In [16]:
# V10 - Fator de demanda
f = 1.2
_, dF1 = simular(terminais_base, cen(horizonte=H30), 1)
_, dF2 = simular(terminais_base, cen(horizonte=H30, fator_demanda=f), 1)
a_, b_ = dF1["notificados"], dF2["notificados"]
mesmas_chegadas = a_[["id_navio", "id_terminal", "t_notificacao"]].equals(b_[["id_navio", "id_terminal", "t_notificacao"]])
_, dF3 = simular(terminais_base, cen(fator_demanda=f), 1)
nf = dF3["notificados"]
ok_f, res = True, []
for _, r in terminais_base.iterrows():
    d_ = nf[nf["id_terminal"] == r["id_terminal"]]
    p = min(1.0, r["prob_compra"] * f)
    frac = d_["comprador"].mean()
    ok_f &= abs(frac - p) <= 3 * np.sqrt(p * (1 - p) / len(d_))
    res.append(f"{r['id_terminal']}: {frac:.3f} (esperado {p:.2f})")
_, dF4 = simular(terminais_base, cen(horizonte=H30, fator_demanda=2.0), 1)
cap = dF4["notificados"]["comprador"].mean() <= 1.0
relatar("V10", "Fator de demanda",
        "Verificar que `fator_demanda` multiplica `prob_compra` (limitado a 1) sem alterar o tráfego de navios, conforme o modelo conceitual.",
        f"(a) Mesma semente com f_d = 1,0 e {f} ({H30} dias); (b) fração de compradores com f_d = {f} em 365 dias; (c) f_d = 2,0, com p·f_d > 1 em T1, T2 e T4.",
        "(a) Mesmos navios e instantes de notificação; (b) fração ≈ min(1, p·f_d) por terminal (3 desvios-padrão); (c) execução sem erro, probabilidade limitada a 1.",
        f"(a) notificações idênticas: {mesmas_chegadas}; (b) " + "; ".join(res) + f"; (c) execução concluída: {bool(cap)}.",
        mesmas_chegadas and ok_f and cap,
        "O fator de demanda aumenta apenas a fração de compradores, como definido no modelo conceitual.")

**V10 – Fator de demanda**

- **Objetivo:** Verificar que `fator_demanda` multiplica `prob_compra` (limitado a 1) sem alterar o tráfego de navios, conforme o modelo conceitual.
- **Configuração:** (a) Mesma semente com f_d = 1,0 e 1.2 (30 dias); (b) fração de compradores com f_d = 1.2 em 365 dias; (c) f_d = 2,0, com p·f_d > 1 em T1, T2 e T4.
- **Esperado:** (a) Mesmos navios e instantes de notificação; (b) fração ≈ min(1, p·f_d) por terminal (3 desvios-padrão); (c) execução sem erro, probabilidade limitada a 1.
- **Obtido:** (a) notificações idênticas: True; (b) T1: 0.821 (esperado 0.84); T2: 0.725 (esperado 0.72); T3: 0.589 (esperado 0.60); T4: 0.804 (esperado 0.78); T5: 0.630 (esperado 0.66); T6: 0.558 (esperado 0.60); (c) execução concluída: True.
- **Conclusão:** Aprovado. O fator de demanda aumenta apenas a fração de compradores, como definido no modelo conceitual.

In [17]:
# V11 - Coerência temporal do log
cols_t = ["t_notificacao", "t_chegada", "t_ini_manobra", "t_ini_operacao", "t_fim_operacao", "t_fim_manobra"]
ordem_ok = all((log_base[a] <= log_base[b]).all() for a, b in zip(cols_t[:-1], cols_t[1:]))
sem_nulos = not log_base.isna().any().any()
fim = origem + pd.Timedelta(days=float(cenario_base["horizonte"]))
dentro = (log_base["t_notificacao"] >= origem).all() and (log_base["t_fim_manobra"] <= fim).all()
conserv = diag_base["n_total_notificados"] == len(log_base) + sum(diag_base["N_final"].values()) + int(diag_base["notificados"]["t_chegada"].isna().sum())
relatar("V11", "Coerência temporal do log e conservação de navios",
        "Verificar que os instantes de cada navio estão em ordem cronológica e dentro do horizonte, e que nenhum navio se perde.",
        "Log do cenário-base; balanço notificados = no log + no porto no fim (N_j) + ainda a caminho.",
        "t_notificacao ≤ t_chegada ≤ t_ini_manobra ≤ t_ini_operacao ≤ t_fim_operacao ≤ t_fim_manobra; sem nulos; dentro do horizonte; balanço fechado.",
        f"Ordem cronológica: {ordem_ok}; sem nulos: {sem_nulos}; dentro do horizonte: {bool(dentro)}; balanço: {diag_base['n_total_notificados']} = {len(log_base)} + {sum(diag_base['N_final'].values())} + {int(diag_base['notificados']['t_chegada'].isna().sum())} ({conserv}).",
        ordem_ok and sem_nulos and dentro and conserv,
        "Os registros seguem a sequência de eventos do modelo conceitual e todos os navios são contabilizados.")

**V11 – Coerência temporal do log e conservação de navios**

- **Objetivo:** Verificar que os instantes de cada navio estão em ordem cronológica e dentro do horizonte, e que nenhum navio se perde.
- **Configuração:** Log do cenário-base; balanço notificados = no log + no porto no fim (N_j) + ainda a caminho.
- **Esperado:** t_notificacao ≤ t_chegada ≤ t_ini_manobra ≤ t_ini_operacao ≤ t_fim_operacao ≤ t_fim_manobra; sem nulos; dentro do horizonte; balanço fechado.
- **Obtido:** Ordem cronológica: True; sem nulos: True; dentro do horizonte: True; balanço: 6467 = 6407 + 23 + 37 (True).
- **Conclusão:** Aprovado. Os registros seguem a sequência de eventos do modelo conceitual e todos os navios são contabilizados.

In [18]:
# V12 - Formato do log_navios.csv
contrato = COLUNAS_LOG
lido = pd.read_csv("saida/log_navios.csv", parse_dates=contrato[5:11])
cols_ok = list(lido.columns) == contrato
tipos_ok = (pd.api.types.is_integer_dtype(lido["rep"]) and pd.api.types.is_integer_dtype(lido["comprador"])
            and pd.api.types.is_float_dtype(lido["volume_t"])
            and all(pd.api.types.is_datetime64_any_dtype(lido[c]) for c in contrato[5:11]))
unico = lido["id_navio"].is_unique and len(lido) == diag_base["n_registrados"]
valores = set(lido["comprador"].unique()) <= {0, 1} and (lido["volume_t"] >= 0).all()
reler = all((lido[c] - log_base[c]).abs().max() <= pd.Timedelta(microseconds=1) for c in contrato[5:11])
relatar("V12", "Formato do `log_navios.csv`",
        "Confirmar que o arquivo gravado segue a definição de saída do modelo conceitual.",
        "`saida/log_navios.csv` do cenário-base, relido do disco.",
        "13 colunas com os nomes e a ordem definidos; instantes em data-hora; `comprador` em {0, 1}; `volume_t` ≥ 0; `id_navio` único; arquivo igual ao log em memória (1 µs).",
        f"Colunas: {cols_ok}; tipos: {tipos_ok}; ids únicos: {unico}; valores válidos: {bool(valores)}; releitura fiel: {bool(reler)}; {len(lido)} linhas.",
        cols_ok and tipos_ok and unico and valores and reler,
        "O log está conforme o modelo conceitual e pode ser lido pelos submodelos seguintes.")

**V12 – Formato do `log_navios.csv`**

- **Objetivo:** Confirmar que o arquivo gravado segue a definição de saída do modelo conceitual.
- **Configuração:** `saida/log_navios.csv` do cenário-base, relido do disco.
- **Esperado:** 13 colunas com os nomes e a ordem definidos; instantes em data-hora; `comprador` em {0, 1}; `volume_t` ≥ 0; `id_navio` único; arquivo igual ao log em memória (1 µs).
- **Obtido:** Colunas: True; tipos: True; ids únicos: True; valores válidos: True; releitura fiel: True; 6407 linhas.
- **Conclusão:** Aprovado. O log está conforme o modelo conceitual e pode ser lido pelos submodelos seguintes.

In [19]:
# V13 - Reprodutibilidade
a1, _ = simular(terminais_base, cen(horizonte=H30), 1)
a2, _ = simular(terminais_base, cen(horizonte=H30), 1)
b, _ = simular(terminais_base, cen(horizonte=H30, semente=int(cenario_base["semente"]) + 1), 1)
c2, _ = simular(terminais_base, cen(horizonte=H30), 2)
ig = a1.equals(a2)
dif_s = not a1.drop(columns="rep").equals(b.drop(columns="rep"))
dif_r = not a1.drop(columns="rep").equals(c2.drop(columns="rep"))
relatar("V13", "Reprodutibilidade e semente",
        "Verificar que a mesma configuração reproduz o mesmo log e que semente e replicação alteram os resultados.",
        f"Quatro execuções de {H30} dias: duas iguais, uma com semente + 1 e uma com a replicação 2.",
        "Logs idênticos para a mesma semente e replicação; diferentes nos outros casos.",
        f"Mesma configuração idêntica: {ig}; outra semente diferente: {dif_s}; outra replicação diferente: {dif_r}.",
        ig and dif_s and dif_r,
        "A semente de `cenario.csv` torna os resultados reproduzíveis.")

**V13 – Reprodutibilidade e semente**

- **Objetivo:** Verificar que a mesma configuração reproduz o mesmo log e que semente e replicação alteram os resultados.
- **Configuração:** Quatro execuções de 30 dias: duas iguais, uma com semente + 1 e uma com a replicação 2.
- **Esperado:** Logs idênticos para a mesma semente e replicação; diferentes nos outros casos.
- **Obtido:** Mesma configuração idêntica: True; outra semente diferente: True; outra replicação diferente: True.
- **Conclusão:** Aprovado. A semente de `cenario.csv` torna os resultados reproduzíveis.

In [20]:
# V14 - Parametrização
unico1 = terminais_base["n_bercos"] == 1
t_b1 = terminais_base.copy()
t_b1.loc[unico1, "taxa_chegada"] *= 1.6
t_b2 = t_b1.copy()
t_b2.loc[unico1, "n_bercos"] = 2
l_b1, _ = simular(t_b1, cen(horizonte=120), 1)
l_b2, _ = simular(t_b2, cen(horizonte=120), 1)
j1 = terminais_base.loc[unico1, "id_terminal"].iloc[0]
e1, e2 = esperas(l_b1[l_b1["id_terminal"] == j1]).mean(), esperas(l_b2[l_b2["id_terminal"] == j1]).mean()
n60, _ = simular(terminais_base, cen(horizonte=60), 1)
n120, _ = simular(terminais_base, cen(horizonte=120), 1)
razao = len(n120) / len(n60)


def literais_numericos(codigo, acc):
    for k in codigo.co_consts:
        if hasattr(k, "co_consts"):
            literais_numericos(k, acc)
        elif isinstance(k, (int, float)) and not isinstance(k, bool):
            acc.add(float(k))
    return acc


literais = set()
for fn in (carregar_parametros, Terminal.__init__, simular):
    literais_numericos(fn.__code__, literais)
permitidos = {0.0, 1.0, 4.0}
relatar("V14", "Parametrização",
        "Verificar que o comportamento muda ao alterar as entradas e que o código não contém valores do cenário.",
        f"(a) Terminal de 1 berço ({j1}) com taxa × 1,6, com 1 e 2 berços; (b) horizonte de 60 e 120 dias; (c) literais numéricos de `carregar_parametros`, `Terminal` e `simular`.",
        "(a) Espera média menor com 2 berços; (b) número de navios aproximadamente proporcional ao horizonte (razão entre 1,7 e 2,3); (c) apenas constantes estruturais (0, 1 e 4 fluxos aleatórios por terminal).",
        f"(a) espera média {e1:.2f} h com 1 berço e {e2:.2f} h com 2; (b) {len(n60)} e {len(n120)} navios (razão {razao:.2f}); (c) literais: {sorted(literais)}.",
        e2 < e1 and 1.7 <= razao <= 2.3 and literais <= permitidos,
        "Berços, taxas e horizonte são controlados pelos arquivos de entrada; o código só tem constantes estruturais.")

**V14 – Parametrização**

- **Objetivo:** Verificar que o comportamento muda ao alterar as entradas e que o código não contém valores do cenário.
- **Configuração:** (a) Terminal de 1 berço (T6) com taxa × 1,6, com 1 e 2 berços; (b) horizonte de 60 e 120 dias; (c) literais numéricos de `carregar_parametros`, `Terminal` e `simular`.
- **Esperado:** (a) Espera média menor com 2 berços; (b) número de navios aproximadamente proporcional ao horizonte (razão entre 1,7 e 2,3); (c) apenas constantes estruturais (0, 1 e 4 fluxos aleatórios por terminal).
- **Obtido:** (a) espera média 18.33 h com 1 berço e 0.99 h com 2; (b) 962 e 2053 navios (razão 2.13); (c) literais: [0.0, 1.0].
- **Conclusão:** Aprovado. Berços, taxas e horizonte são controlados pelos arquivos de entrada; o código só tem constantes estruturais.

In [21]:
# V15 - Utilização dos berços versus teoria de filas
H = float(cenario_base["horizonte"]) * HORAS_POR_DIA
ok_u, res = True, []
for _, r in terminais_base.iterrows():
    teo = r["taxa_chegada"] * ((r["op_p1"] + r["op_p2"] + r["op_p3"]) / 3 + 2 * r["tempo_manobra"]) / r["n_bercos"]
    d_ = log_base[log_base["id_terminal"] == r["id_terminal"]]
    obs = horas(d_["t_fim_manobra"] - d_["t_ini_manobra"]).sum() / (r["n_bercos"] * H)
    ok_u &= abs(obs - teo) <= 0.05
    res.append(f"{r['id_terminal']}: observada {obs:.3f}, teórica {teo:.3f}")
relatar("V15", "Utilização dos berços versus teoria",
        "Comparar a utilização simulada com a de um sistema M/G/c estável, ρ = λ·(E[operação] + 2·manobra)/c.",
        "Log do cenário-base; utilização = tempo total de ocupação dos berços ÷ (c × horizonte).",
        "Diferença ≤ 0,05 por terminal (pequeno viés para baixo pelas 48 h iniciais sem chegadas e pelos navios em curso no fim).",
        "; ".join(res) + ".",
        ok_u,
        "A carga simulada é coerente com a calculada analiticamente a partir dos parâmetros.")

**V15 – Utilização dos berços versus teoria**

- **Objetivo:** Comparar a utilização simulada com a de um sistema M/G/c estável, ρ = λ·(E[operação] + 2·manobra)/c.
- **Configuração:** Log do cenário-base; utilização = tempo total de ocupação dos berços ÷ (c × horizonte).
- **Esperado:** Diferença ≤ 0,05 por terminal (pequeno viés para baixo pelas 48 h iniciais sem chegadas e pelos navios em curso no fim).
- **Obtido:** T1: observada 0.623, teórica 0.648; T2: observada 0.716, teórica 0.700; T3: observada 0.701, teórica 0.688; T4: observada 0.865, teórica 0.859; T5: observada 0.565, teórica 0.590; T6: observada 0.513, teórica 0.493.
- **Conclusão:** Aprovado. A carga simulada é coerente com a calculada analiticamente a partir dos parâmetros.

In [22]:
resumo_testes = pd.DataFrame(RESULTADOS)
display(resumo_testes)
print("Testes aprovados:", int((resumo_testes["Resultado"] == "Aprovado").sum()), "de", len(resumo_testes))

,Teste,Descrição,Resultado
0,V01,Arquivos de entrada,Aprovado
1,V02,Geração de navios,Aprovado
2,V03,Relação notificação → chegada,Aprovado
3,V04,Capacidade dos berços,Aprovado
4,V05,Disciplina FIFO,Aprovado
5,V06,Manobras e ocupação do berço,Aprovado
6,V07,Operação no terminal,Aprovado
7,V08,Geração de compradores,Aprovado
8,V09,Geração da demanda (volume),Aprovado
9,V10,Fator de demanda,Aprovado


Testes aprovados: 15 de 15


## 6. Referências

- Costa, M. G.; Mesquita, M. A. (2023). *Minimizing total tardiness and makespan in ship fuelling operations in a maritime terminal*. Manuscrito.
- Grupo G112 (2026). *Projeto BVE - Modelo Conceitual*, versão 1.1; *Proposta de Consultoria*; *Solicitação de Dados*.
- Law, A. M. (2013). A tutorial on how to select simulation input probability distributions. *Proceedings of the 2013 Winter Simulation Conference*.
- PRO3342 (2026). *D07 - BVE: Modelo Conceitual dos Navios* (modelo de referência, versão 1.0) e *D08 - BVE: Modelo Computacional dos Navios* (notebook-modelo e dados do cenário-base).
- SimPy 4, NumPy e pandas: documentação oficial.